# Quick Demo: Robotic Bioprinting Pipeline

This notebook shows you what the system does — step by step, with pictures.

**No GPU needed.** Everything runs on your laptop in under 1 minute.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## Step 1: Generate a Fake Wound

We create a synthetic wound image — a blob-shaped region that looks like a wound.
The AI will learn to detect these boundaries.

In [ ]:
from data.synthetic_generator import generate_star_convex_wound
from data.polar_conversion import polar_to_cartesian

# Generate 4 random wound samples
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

for i, ax in enumerate(axes):
    np.random.seed(i * 13 + 7)
    sample = generate_star_convex_wound()
    
    ax.imshow(sample['image'])
    
    # Draw the boundary (ground truth)
    pts = sample['points'] * 256
    pts_closed = np.vstack([pts, pts[0]])
    ax.plot(pts_closed[:, 0], pts_closed[:, 1], 'lime', linewidth=2)
    ax.plot(sample['centroid'][0]*256, sample['centroid'][1]*256, 'r+', markersize=15, markeredgewidth=2)
    ax.set_title(f'Wound {i+1}', fontsize=11)
    ax.axis('off')

plt.suptitle('Synthetic Wound Images with Ground Truth Boundaries (green)', fontsize=13)
plt.tight_layout()
plt.show()

print('The GREEN line is what the AI needs to learn to predict.')
print('The RED cross is the wound center.')

## Step 2: How We Represent Boundaries (Polar Coordinates)

Instead of random scattered points, we use **polar coordinates**:
- Pick the center of the wound
- Measure how far the edge is in every direction (like a radar)

This guarantees the boundary is always **closed** and **ordered** (no crossings!).

In [ ]:
np.random.seed(42)
sample = generate_star_convex_wound()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left: the wound image
ax1.imshow(sample['image'])
pts = sample['points'] * 256
cx, cy = sample['centroid'] * 256
ax1.plot(cx, cy, 'r+', markersize=15, markeredgewidth=2)
pts_closed = np.vstack([pts, pts[0]])
ax1.plot(pts_closed[:, 0], pts_closed[:, 1], 'lime', linewidth=2)

# Draw "radar beams" from center to edge
for j in range(0, len(pts), 8):
    ax1.plot([cx, pts[j, 0]], [cy, pts[j, 1]], 'yellow', linewidth=0.5, alpha=0.6)
ax1.set_title('Polar Representation\n(radar beams from center to edge)', fontsize=11)
ax1.axis('off')

# Right: polar plot (angle vs radius)
angles_deg = np.linspace(0, 360, len(sample['radii']), endpoint=False)
ax2.bar(angles_deg, sample['radii'] * 256, width=4, color='coral', alpha=0.7)
ax2.set_xlabel('Angle (degrees)', fontsize=11)
ax2.set_ylabel('Distance to edge (pixels)', fontsize=11)
ax2.set_title('What the AI actually predicts:\n"How far is the edge in each direction?"', fontsize=11)
ax2.set_xlim(0, 360)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Number of directions measured: {len(sample["radii"])}')
print('This is all the AI needs to output — one center + 64 distances.')

## Step 3: The Neural Network (CNN-Transformer)

Our AI has two parts:
1. **CNN (ResNet-50)** — looks at small patterns in the image (edges, colors, textures)
2. **Transformer** — understands the big picture (how patterns relate to each other)

Both are built **from scratch** using basic PyTorch operations.

In [ ]:
import torch
from models.encoder import CNNTransformerEncoder
from models.polar_decoder import PolarDecoder

# Create the model (small version for demo — full version trains on GPU)
encoder = CNNTransformerEncoder(d_model=64, num_heads=4, num_layers=2, pretrained=False)
decoder = PolarDecoder(d_model=64, num_radii=64)

# Count parameters
enc_params = sum(p.numel() for p in encoder.parameters())
dec_params = sum(p.numel() for p in decoder.parameters())

print(f'Encoder parameters: {enc_params:,}')
print(f'Decoder parameters: {dec_params:,}')
print(f'Total: {enc_params + dec_params:,}')
print()

# Run a fake image through the model
fake_image = torch.randn(1, 3, 256, 256)  # 1 random RGB image, 256x256 pixels
with torch.no_grad():
    features = encoder(fake_image)      # (1, 64, 64) — 64 feature tokens
    prediction = decoder(features)      # dict with centroid, radii, points

print(f'Input: image of shape {list(fake_image.shape)}')
print(f'Encoder output: {list(features.shape)} (64 feature tokens of size 64)')
print(f'Predicted centroid: {prediction["centroid"][0].numpy().round(3)}')
print(f'Predicted radii: {prediction["radii"].shape} -> {prediction["radii"][0][:5].numpy().round(3)}...')
print(f'Predicted boundary points: {prediction["points"].shape}')

## Step 4: From Wound Boundary to Robot Path

Once we know the wound shape, we need to plan how the robot fills it:

1. Load the 3D scaffold (the wound holder)
2. Detect where the void (wound) is
3. Generate a honeycomb filling pattern
4. Optimize the order to visit cells (shortest path = less wasted movement)

In [ ]:
from modules.stl_analysis import analyze_scaffold
from modules.honeycomb import create_hex_grid, hexagon_perimeter, compute_grid_params

# Load and analyze the scaffold
scaffold = analyze_scaffold('../data/scaffold_curved_void.stl')

cyl = scaffold['cylinder']
vb = scaffold['void_bounds']

print('=== Scaffold Analysis ===')
print(f'  Shape: semi-cylinder, radius = {cyl["radius"]:.1f} mm')
print(f'  Void detected: {vb["void_width"]:.1f} mm wide x {vb["void_length"]:.1f} mm long')
print(f'  Shell thickness: {vb["shell_thickness"]:.1f} mm')
print()

# Generate honeycomb grid
nx, ny, hex_side = compute_grid_params(vb['void_width'], vb['void_length'])
grid = create_hex_grid(nx, ny, hex_side)

print(f'=== Honeycomb Plan ===')
print(f'  Grid: {nx} x {ny} = {nx*ny} hexagonal cells')
print(f'  Each cell side: {hex_side:.1f} mm')
print(f'  Layers to fill: {int(np.ceil(vb["shell_thickness"] / 0.4))}')

In [ ]:
# Visualize the honeycomb grid
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: honeycomb pattern in 2D (UV space)
for iy in range(ny):
    for ix in range(nx):
        center = grid[iy, ix]
        pts = hexagon_perimeter(center, hex_side, n_per_edge=10)
        pts_closed = np.vstack([pts, pts[0]])
        ax1.plot(pts_closed[:, 0], pts_closed[:, 1], 'b-', linewidth=1.2)
        ax1.plot(center[0], center[1], 'r.', markersize=5)

ax1.set_xlabel('U (mm) — arc length')
ax1.set_ylabel('V (mm) — axial')
ax1.set_title('Honeycomb Pattern (flat, before mapping to surface)')
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.2)

# Right: the scaffold with void highlighted
vertices = scaffold['vertices']
void_pts = vertices[scaffold['void_vid']]

# 2D projection (Y vs Z — curvature plane)
ax2.scatter(vertices[::5, 1], vertices[::5, 2], s=0.5, c='gray', alpha=0.3, label='Scaffold')
ax2.scatter(void_pts[:, 1], void_pts[:, 2], s=3, c='red', alpha=0.7, label='Detected void')

# Draw fitted circle
theta_plot = np.linspace(-np.pi/2, np.pi/2, 100)
ax2.plot(cyl['cy'] + cyl['radius']*np.sin(theta_plot),
         cyl['cz'] + cyl['radius']*np.cos(theta_plot), 'g--', linewidth=1.5, label=f'Fitted circle R={cyl["radius"]:.0f}mm')
ax2.set_xlabel('Y (mm)')
ax2.set_ylabel('Z (mm)')
ax2.set_title('Scaffold Cross-Section (curvature plane)')
ax2.legend(fontsize=9)
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print('Left: the beehive pattern we will print (flat layout)')
print('Right: the curved scaffold where it will be printed (red = where the wound is)')

## Step 5: Map Honeycomb onto the Curved Surface

The wound isn't flat — it's on a curved body part. So we **conformally map** our flat honeycomb pattern onto the curved surface. Think of it like wrapping gift paper around a tube.

In [ ]:
from modules.trajectory_planner import plan_full_trajectory

# Generate the full robot trajectory
traj_result = plan_full_trajectory(
    void_bounds=vb,
    cyl_radius=cyl['radius'],
    cyl_cy=cyl['cy'],
    cyl_cz=cyl['cz'],
    rise=20.0,
    optimize_tsp=True,
)

print(f'\nTotal robot trajectory: {traj_result["n_points"]:,} points')
print(f'That is {traj_result["n_points"] * 0.05:.0f} seconds of robot motion at 20 Hz')

In [ ]:
# Visualize the 3D trajectory
traj_xyz = traj_result['traj_xyz_mm']
traj_uv = traj_result['traj_uv']

# Only show deposition points (not travel moves)
dep_mask = traj_uv[2] <= 0  # h <= 0 means ON or INSIDE the surface
travel_mask = traj_uv[2] > 0

fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(traj_xyz[0, dep_mask], traj_xyz[1, dep_mask], traj_xyz[2, dep_mask],
           s=0.3, c='blue', alpha=0.5, label='Deposition')
ax1.scatter(traj_xyz[0, travel_mask][::10], traj_xyz[1, travel_mask][::10], traj_xyz[2, travel_mask][::10],
           s=0.1, c='gray', alpha=0.2, label='Travel')
ax1.set_xlabel('X (mm)')
ax1.set_ylabel('Y (mm)')
ax1.set_zlabel('Z (mm)')
ax1.set_title('3D Robot Trajectory')
ax1.legend()
ax1.view_init(25, 135)

ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(traj_xyz[0, dep_mask], traj_xyz[1, dep_mask], traj_xyz[2, dep_mask],
           s=0.5, c='blue', alpha=0.6)
ax2.set_xlabel('X (mm)')
ax2.set_ylabel('Y (mm)')
ax2.set_zlabel('Z (mm)')
ax2.set_title('Just the Deposited Material')
ax2.view_init(0, 90)

plt.tight_layout()
plt.show()

print('Blue = where the robot prints material')
print('Gray = where the robot travels without printing')
print('Notice: the honeycomb is CURVED — it follows the body surface!')

## Step 6: The Robot Arm

Our robot has **8 joints** (degrees of freedom):
- 2 prismatic joints (XY gantry — moves the whole arm left/right/forward/back)
- 6 revolute joints (the arm itself — like your shoulder, elbow, wrist)

For each point in the trajectory, we solve **inverse kinematics** = "what angles should each joint be at so the tip reaches this position?"

In [ ]:
from modules.robot_model import forward_kinematics_8dof, home_configuration, JOINT_NAMES

# Show robot at home position
q_home = home_configuration()
T_home = forward_kinematics_8dof(q_home)

print('Robot at home position (all joints = 0):')
print(f'  End-effector (tip) is at: X={T_home[0,3]*1000:.1f}mm, Y={T_home[1,3]*1000:.1f}mm, Z={T_home[2,3]*1000:.1f}mm')
print()
print('Joint names and types:')
for i, name in enumerate(JOINT_NAMES):
    jtype = 'prismatic (slides)' if i < 2 else 'revolute (rotates)'
    print(f'  Joint {i+1}: {name} — {jtype}')

In [ ]:
from modules.inverse_kinematics import InverseKinematicsSolver, IKParams

# Solve IK for just 20 points (to keep the demo fast)
params = IKParams()
params.max_iter = 30
solver = InverseKinematicsSolver(params)

# Take a small slice of the trajectory
n_demo = 20
step = max(1, traj_result['n_points'] // n_demo)
demo_indices = np.arange(0, traj_result['n_points'], step)[:n_demo]

traj_demo = traj_result['traj_m'][:, demo_indices]
R_demo = traj_result['R_targets'][:, :, demo_indices]

print(f'Solving inverse kinematics for {n_demo} sample points...')
ik_result = solver.solve_trajectory(traj_demo, R_demo)

print(f'\nResults:')
print(f'  Average error: {ik_result["errors"].mean()*1000:.2f} mm')
print(f'  Max error: {ik_result["errors"].max()*1000:.2f} mm')
print(f'  Points needing advanced solving: {(ik_result["phases"] > 0).sum()}/{n_demo}')

## Summary

That's the whole pipeline!

| Step | What happens | Module |
|------|-------------|--------|
| 1 | Camera takes a photo | (hardware) |
| 2 | AI detects wound boundary | `models/` |
| 3 | System builds 3D surface | `modules/stl_analysis` |
| 4 | Honeycomb fill pattern generated | `modules/honeycomb` |
| 5 | Pattern mapped to curved surface | `modules/conformal_mapping` |
| 6 | Shortest visitation order found | `modules/tsp_solver` |
| 7 | Robot joint angles computed | `modules/inverse_kinematics` |
| 8 | Robot moves and prints | CoppeliaSim |

### Next Steps
- **Train the AI**: Run `notebooks/01_ablation_study_kaggle.ipynb` on Kaggle (needs GPU)
- **Full IK on trajectory**: Remove the `n_demo=20` limit above for the full 23,000+ points
- **CoppeliaSim**: Connect to the physics simulator for realistic robot animation

In [ ]:
print('Done! You just saw the entire bioprinting pipeline in action.')
print('Questions? Check the README.md or the other notebooks.')